In [1]:
import pandas as pd
import ollama
import time

from tqdm import tqdm

In [2]:
INPUT_PATH = "../data/processed/arxiv_with_readability.csv"

# Load dataset
_df = pd.read_csv(INPUT_PATH)

# Remove missing abstracts
_df = _df.dropna(subset=["abstract"])

print(f"Dataset size: {_df.shape[0]}")

_df.head(3)

Dataset size: 2272


,paper_id,title,field,abstract,published,year,arxiv_id_full,citation_count,flesch_reading_ease,flesch_kincaid_grade,gunning_fog,smog_index
0,http://arxiv.org/abs/1901.00064v3,Impossibility and Uncertainty Theorems in AI V...,computer_science,Utility functions or their equivalents (value ...,2018-12-31 23:51:27+00:00,2018,1901.00064,54,9.464828,20.172783,23.084729,19.430817
1,http://arxiv.org/abs/1901.00063v2,Extreme Relative Pose Estimation for RGB-D Sca...,computer_science,Estimating the relative rigid pose between two...,2018-12-31 23:43:16+00:00,2018,1901.00063,60,16.174642,16.375542,20.441908,17.693802
2,http://arxiv.org/abs/1901.00062v3,Deep Frame Prediction for Video Coding,computer_science,We propose a novel frame prediction method usi...,2018-12-31 23:41:50+00:00,2018,1901.00062,66,36.931731,12.952436,15.594872,14.554593


In [3]:
# SMALL TEST SAMPLE
sample_df = _df.sample(150, random_state=42).copy()

print(sample_df.shape)

(150, 12)


In [4]:
PROMPT_TEMPLATE = """
Rewrite the following scientific abstract in your own way!(approximately same word count if you want)
Abstract:
{abstract}
"""

In [5]:
MODEL_NAME = "tinyllama"

In [6]:
def rewrite_abstract(text):

    try:
        prompt = PROMPT_TEMPLATE.format(abstract=text)

        response = ollama.chat(
            model=MODEL_NAME,
            messages=[
                {
                    "role": "user",
                    "content": prompt
                }
            ]
        )

        rewritten = response["message"]["content"]

        return rewritten.strip()

    except Exception as e:
        print("ERROR:", e)
        return None 

In [7]:
ai_abstracts = []

for abstract in tqdm(sample_df["abstract"]):

    rewritten = rewrite_abstract(abstract)

    ai_abstracts.append(rewritten)

100%|██████████| 150/150 [1:54:24<00:00, 45.76s/it]  


In [8]:
sample_df["ai_abstract"] = ai_abstracts

sample_df[["abstract", "ai_abstract"]].head(3)

,abstract,ai_abstract
188,"In this paper, we propose an adaptive beam tha...","Scientific Abstract:\nIn this paper, we propos..."
1803,We investigate the possibility of a semantic a...,Abstract:\nWe investigate the possibility of a...
2119,The incompressible 2D Euler equations on a sph...,Rewrite the scientific abstract above as follo...


In [9]:
OUTPUT_PATH = "../data/processed/arxiv_ai_rewritten.csv"

sample_df.to_csv(OUTPUT_PATH, index=False)

print(f"Saved rewritten dataset to {OUTPUT_PATH}")

Saved rewritten dataset to ../data/processed/arxiv_ai_rewritten.csv
